In [4]:
!pip3 install tensorboard

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 5.5 MB 2.8 MB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 21.6 MB/s eta 0:00:01
     |████████████████████████████████| 107 kB 30.8 MB/s eta 0:00:01
     |████████████████████████████████| 427 kB 24.5 MB/s eta 0:00:01
     |████████████████████████████████| 11.8 MB 19.1 MB/s eta 0:00:01
     |████████████████████████████████| 224 kB 115.3 MB/s eta 0:00:01
     |████████████████████████████████| 4.7 MB 27.5 MB/s eta 0:00:01
     |████████████████████████████████| 135 kB 23.6 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [5]:
%load_ext tensorboard

In [6]:
! rm -rf ./logs/ 

In [9]:
from datetime import datetime
from packaging import version

import tensorflow as tf
from tensorflow import keras
tf.debugging.experimental.enable_dump_debug_info('./logs/',
                                                 tensor_debug_mode="FULL_HEALTH", 
                                                 circular_buffer_size=-1)
from keras import backend as K
import numpy as np

print("TensorFlow version: ", tf.__version__)
assert version.parse(tf.__version__).release[0] >= 2, \
    "This notebook requires TensorFlow 2.0 or above."

/Users/rishirajkuleri/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


INFO:tensorflow:Enabled dumping callback in thread MainThread (dump root: ./logs/, tensor debug mode: FULL_HEALTH)
TensorFlow version:  2.20.0


In [8]:
!pip3 install tensorflow

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 200.4 MB 35.7 MB/s eta 0:00:011
     |████████████████████████████████| 71 kB 3.0 MB/s  eta 0:00:01
     |████████████████████████████████| 2.8 MB 20.8 MB/s eta 0:00:01
     |████████████████████████████████| 25.8 MB 8.7 MB/s eta 0:00:01
     |████████████████████████████████| 663 kB 34.8 MB/s eta 0:00:01
     |████████████████████████████████| 1.4 MB 80.2 MB/s eta 0:00:01
     |████████████████████████████████| 57 kB 19.1 MB/s eta 0:00:01
     |████████████████████████████████| 64 kB 16.9 MB/s eta 0:00:01
     |████████████████████████████████| 61 kB 1.2 MB/s  eta 0:00:01
     |████████████████████████████████| 330 kB 14.1 MB/s eta 0:00:01
     |████████████████████████████████| 243 kB 112.8 MB/s eta 0:00:01
     |████████████████████████████████| 159 kB 49.7 MB/s eta 0:00:01
     |████████████████████████████████| 71 kB 24.1 MB/s eta 0:00:01
     |████████████████████

In [10]:
data_size = 1000
# 80% of the data is for training.
train_pct = 0.8

train_size = int(data_size * train_pct)

# Create some input data between -1 and 1 and randomize it.
x = np.linspace(-1, 1, data_size)
np.random.shuffle(x)

# Generate the output data.
# y = 0.6x + 4 + noise
y = 0.6 * x + 4 + np.random.normal(0, 0.07, (data_size, ))

# Split into test and train pairs.
x_train, y_train = x[:train_size], y[:train_size]
x_test, y_test = x[train_size:], y[train_size:]

In [ ]:
logdir = "logs/scalars/" + datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = keras.callbacks.TensorBoard(log_dir=logdir)

# Model architecture with capacity and regularization
model = keras.models.Sequential([
    keras.layers.Dense(64, activation='relu', input_dim=1, 
                       kernel_regularizer=keras.regularizers.l2(0.001)),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(32, activation='relu',
                       kernel_regularizer=keras.regularizers.l2(0.001)),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1),
])

# Better optimizer with adaptive learning rate
model.compile(
    loss='mse',
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    metrics=['mae', 'mse']  # Track both Mean Absolute Error and MSE
)

# Add early stopping and learning rate reduction callbacks
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

print("Training... This may take a bit longer with the improved model.")
training_history = model.fit(
    x_train,
    y_train,
    batch_size=32,  # Smaller batch size for better generalization
    verbose=1,
    epochs=50,
    validation_data=(x_test, y_test),
    callbacks=[tensorboard_callback, early_stopping, reduce_lr],
)

# Better evaluation metrics
print("=" * 50)
print("Training Results:")
print(f"Final train loss (MSE): {training_history.history['loss'][-1]:.4f}")
print(f"Final validation loss (MSE): {training_history.history['val_loss'][-1]:.4f}")
print(f"Final train MAE: {training_history.history['mae'][-1]:.4f}")
print(f"Final validation MAE: {training_history.history['val_mae'][-1]:.4f}")
print(f"Best validation loss: {min(training_history.history['val_loss']):.4f}")
print(f"Epochs trained: {len(training_history.history['loss'])}")
print("=" * 50)

Training... This may take a bit longer with the improved model.
Epoch 1/50


/Users/rishirajkuleri/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 15.7681 - mae: 3.9513 - mse: 15.7236 - val_loss: 12.8723 - val_mae: 3.5616 - val_mse: 12.8286 - learning_rate: 0.0010
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 11.8212 - mae: 3.4019 - mse: 11.7774 - val_loss: 6.8067 - val_mae: 2.5389 - val_mse: 6.7622 - learning_rate: 0.0010
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 5.2656 - mae: 2.1589 - mse: 5.2206 - val_loss: 0.9864 - val_mae: 0.8580 - val_mse: 0.9396 - learning_rate: 0.0010
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.1197 - mae: 0.8645 - mse: 1.0727 - val_loss: 0.5395 - val_mae: 0.6205 - val_mse: 0.4935 - learning_rate: 0.0010
Epoch 5/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.8568 - mae: 0.7479 - mse: 0.8112 - val_loss: 0.3829 - val_mae: 0.5219 - val_mse: 0.3384 - learning_rate: 0.0010
Epoch 6/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.6983 - mae: 0.6565 - mse: 0.6540 - val_loss: 0.2928 - val_mae: 0.4447 - val_mse: 0.2

In [ ]:
%tensorboard --logdir logs/

In [ ]:
#http://localhost:6006

#The visulaizations and dashboards are created within this site. 